Link two datasets
=================

Introduction
------------

This example shows how two datasets with data about persons can be
linked. We will try to link the data based on attributes like first
name, surname, sex, date of birth, place and address. The data used in
this example is part of
[Febrl](https://sourceforge.net/projects/febrl/) and is fictitious.

First, start with importing the ``recordlinkage`` module. The submodule
``recordlinkage.datasets`` contains several datasets that can be used
for testing. For this example, we use the Febrl datasets 4A and 4B.
These datasets can be loaded with the function ``load_febrl4``.

In [1]:
!pip install recordlinkage

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.9/926.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 12.1 MB/s eta 0:00:00


In [2]:
import recordlinkage
from recordlinkage.datasets import load_febrl4

The datasets are loaded with the following code. The returned datasets
are of type ``pandas.DataFrame``. This makes it easy to manipulate the
data if desired. For details about data manipulation with ``pandas``,
see their comprehensive documentation http://pandas.pydata.org/.

In [3]:
dfA, dfB = load_febrl4()
dfA

,given_name,surname,street_number,address_1,address_2,suburb,postcode,state,date_of_birth,soc_sec_id
rec_id,,,,,,,,,,
rec-1070-org,michaela,neumann,8,stanley street,miami,winston hills,4223,nsw,19151111,5304218
rec-1016-org,courtney,painter,12,pinkerton circuit,bega flats,richlands,4560,vic,19161214,4066625
rec-4405-org,charles,green,38,salkauskas crescent,kela,dapto,4566,nsw,19480930,4365168
rec-1288-org,vanessa,parr,905,macquoid place,broadbridge manor,south grafton,2135,sa,19951119,9239102
rec-3585-org,mikayla,malloney,37,randwick road,avalind,hoppers crossing,4552,vic,19860208,7207688
...,...,...,...,...,...,...,...,...,...,...
rec-2153-org,annabel,grierson,97,mclachlan crescent,lantana lodge,broome,2480,nsw,19840224,7676186
rec-1604-org,sienna,musolino,22,smeaton circuit,pangani,mckinnon,2700,nsw,19890525,4971506
rec-1003-org,bradley,matthews,2,jondol place,horseshoe ck,jacobs well,7018,sa,19481122,8927667


Preprocessing
-----------------
Although it is not neccessary, because the ``febr`` dataset is already clean, ``clean`` and ``phonetic`` options are applied as an illustration of their usage.

In [6]:
type(dfA)

pandas.core.frame.DataFrame

In [4]:
from recordlinkage.preprocessing import clean, phonetic
dfA['address_1_cleaned'] = clean(dfA['address_1'], replace_by_none='[^ \\-\\_\(\)A-Za-z0-9]+',
                    remove_brackets=True, strip_accents = 'unicode')


dfA['given_name_methaphone'] = phonetic(dfA['given_name'], method='metaphone')
dfA['given_name_nysiis'] = phonetic(dfA['given_name'], method='nysiis')

dfB['address_1_cleaned'] = clean(dfB['address_1'], replace_by_none='[^ \\-\\_\(\)A-Za-z0-9]+',
                    remove_brackets=True, strip_accents = 'unicode')


dfB['given_name_methaphone'] = phonetic(dfB['given_name'], method='metaphone')
dfB['given_name_nysiis'] = phonetic(dfB['given_name'], method='nysiis')
dfB

<>:2: SyntaxWarning: invalid escape sequence '\('
<>:9: SyntaxWarning: invalid escape sequence '\('
<>:2: SyntaxWarning: invalid escape sequence '\('
<>:9: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_376/2296246855.py:2: SyntaxWarning: invalid escape sequence '\('
  dfA['address_1_cleaned'] = clean(dfA['address_1'], replace_by_none='[^ \\-\\_\(\)A-Za-z0-9]+',
/tmp/ipykernel_376/2296246855.py:9: SyntaxWarning: invalid escape sequence '\('
  dfB['address_1_cleaned'] = clean(dfB['address_1'], replace_by_none='[^ \\-\\_\(\)A-Za-z0-9]+',


,given_name,surname,street_number,address_1,address_2,suburb,postcode,state,date_of_birth,soc_sec_id,address_1_cleaned,given_name_methaphone,given_name_nysiis
rec_id,,,,,,,,,,,,,
rec-561-dup-0,elton,NaN,3,light setreet,pinehill,windermere,3212,vic,19651013,1551941,light setreet,ELTN,ELTAN
rec-2642-dup-0,mitchell,maxon,47,edkins street,lochaoair,north ryde,3355,nsw,19390212,8859999,edkins street,MXL,MATCAL
rec-608-dup-0,NaN,white,72,lambrigg street,kelgoola,broadbeach waters,3159,vic,19620216,9731855,lambrigg street,NaN,NaN
rec-3239-dup-0,elk i,menzies,1,lyster place,NaN,northwood,2585,vic,19980624,4970481,lyster place,ELK,ELC
rec-2886-dup-0,NaN,garanggar,NaN,may maxwell crescent,springettst arcade,forest hill,2342,vic,19921016,1366884,may maxwell crescent,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
rec-4495-dup-0,connor,belperio,15,NaN,NaN,ryde,2570,nsw,19170518,5394641,NaN,KNR,CANAR
rec-4211-dup-0,daniel,maspn,9,derrington crescent,el pedro caravan park,sunnybank,4350,vic,19500705,5525378,derrington crescent,TNL,DANAL
rec-3131-dup-0,samuel,crofs,613,banjine street,kurrajong vlge,pengzin,2230,qld,19410531,4467228,banjine street,SML,SANAL


Make record pairs
-----------------

It is very intuitive to compare each record in DataFrame ``dfA`` with
all records of DataFrame ``dfB``. In fact, we want to make record pairs.
Each record pair should contain one record of ``dfA`` and one record of
``dfB``. This process of making record pairs is also called "indexing".
With the ``recordlinkage`` module, indexing is easy. First, load the
``index.Index`` class and call the ``.full`` method. This object
generates a full index on a ``.index(...)`` call. In case of
deduplication of a single dataframe, one dataframe is sufficient as
argument.

In [7]:
indexer = recordlinkage.Index()
indexer.full()
pairs = indexer.index(dfA, dfB)
pairs

MultiIndex([('rec-1070-org',  'rec-561-dup-0'),
            ('rec-1070-org', 'rec-2642-dup-0'),
            ('rec-1070-org',  'rec-608-dup-0'),
            ('rec-1070-org', 'rec-3239-dup-0'),
            ('rec-1070-org', 'rec-2886-dup-0'),
            ('rec-1070-org', 'rec-4285-dup-0'),
            ('rec-1070-org',  'rec-929-dup-0'),
            ('rec-1070-org', 'rec-4833-dup-0'),
            ('rec-1070-org',  'rec-717-dup-0'),
            ('rec-1070-org', 'rec-3984-dup-0'),
            ...
            (  'rec-66-org',  'rec-670-dup-0'),
            (  'rec-66-org', 'rec-4134-dup-0'),
            (  'rec-66-org', 'rec-3866-dup-0'),
            (  'rec-66-org', 'rec-3152-dup-0'),
            (  'rec-66-org', 'rec-3363-dup-0'),
            (  'rec-66-org', 'rec-4495-dup-0'),
            (  'rec-66-org', 'rec-4211-dup-0'),
            (  'rec-66-org', 'rec-3131-dup-0'),
            (  'rec-66-org', 'rec-3815-dup-0'),
            (  'rec-66-org',  'rec-493-dup-0')],
           names=['rec_

With the method ``index``, all possible (and unique) record pairs are
made. The method returns a ``pandas.MultiIndex``. The number of pairs is
equal to the number of records in ``dfA`` times the number of records in
``dfB``.

In [ ]:
print (len(dfA), len(dfB), len(pairs))

5000 5000 25000000


Many of these record pairs do not belong to the same person. In case of
one-to-one matching, the number of matches should be no more than the
number of records in the smallest dataframe. In case of full indexing,
``min(len(dfA), len(N_dfB))`` is much smaller than ``len(pairs)``. The
``recordlinkage`` module has some more advanced indexing methods to
reduce the number of record pairs. Obvious non-matches are left out of
the index. Note that if a matching record pair is not included in the
index, it can not be matched anymore.

One of the most well known indexing methods is named *blocking*. This
method includes only record pairs that are identical on one or more
stored attributes of the person (or entity in general). The blocking
method can be used in the ``recordlinkage`` module.

In [8]:
from recordlinkage.index import Block
indexer = recordlinkage.Index()
#Add Block algorithm with several columns
#Very restrictive
#indexer.add(Block(['given_name','surname'], ['given_name','surname']))

#Add Several results of block
#indexer.add(Block('given_name', 'given_name'))
#indexer.add(Block('surname', 'surname'))
#or
#indexer.block('given_name', 'given_name')
#indexer.block('surname', 'surname')

indexer.block('given_name', 'given_name')

candidate_links = indexer.index(dfA, dfB)
len(candidate_links)

77249

The argument "given\_name" is the blocking variable. This variable has
to be the name of a column in ``dfA`` and ``dfB``. It is possible to
parse a list of columns names to block on multiple variables. Blocking
on multiple variables will reduce the number of record pairs even
further.

Another implemented indexing method is *Sorted Neighbourhood Indexing*
(``recordlinkage.index.SortedNeighbourhood``). This method is very
useful when there are many misspellings in the string were used for
indexing. In fact, sorted neighbourhood indexing is a generalisation of
blocking. See the documentation for details about sorted neighbourd
indexing.

Compare records
---------------

Each record pair is a candidate match. To classify the candidate record
pairs into matches and non-matches, compare the records on all
attributes both records have in common. The ``recordlinkage`` module has
a class named ``Compare``. This class is used to compare the records.
The following code shows how to compare attributes.

In [ ]:
compare_cl = recordlinkage.Compare()
compare_cl.exact("given_name", "given_name", label="given_name")
compare_cl.string("surname", "surname", method="jarowinkler", threshold=0.85, label="surname")
compare_cl.exact("date_of_birth", "date_of_birth", label="date_of_birth")
compare_cl.exact("suburb", "suburb", label="suburb")
compare_cl.exact("state", "state", label="state")
compare_cl.string("address_1", "address_1", threshold=0.85, label="address_1")
features = compare_cl.compute(candidate_links, dfA, dfB)

The comparing of record pairs starts when the ``compute`` method is
called. All attribute comparisons are stored in a DataFrame with
horizontally the features and vertically the record pairs.

In [ ]:
features

given_name  surname  date_of_birth  suburb  \
rec_id_1     rec_id_2                                                     
rec-1070-org rec-3024-dup-0           1      0.0              0       0   
             rec-2371-dup-0           1      0.0              0       0   
             rec-4652-dup-0           1      0.0              0       0   
             rec-4795-dup-0           1      0.0              0       0   
             rec-1314-dup-0           1      0.0              0       0   
...                                 ...      ...            ...     ...   
rec-1003-org rec-3321-dup-0           1      0.0              0       0   
rec-4883-org rec-4883-dup-0           1      1.0              1       1   
rec-66-org   rec-3318-dup-0           1      0.0              0       0   
             rec-66-dup-0             1      1.0              1       1   
             rec-3838-dup-0           1      0.0              0       0   

                             state  address_1  
rec_id_1     rec_id_2                          
rec-1070-org rec-3024-dup-0      1        0.0  
             rec-2371-dup-0      0        0.0  
             rec-4652-dup-0      0        0.0  
             rec-4795-dup-0      1        0.0  
             rec-1314-dup-0      1        0.0  
...                            ...        ...  
rec-1003-org rec-3321-dup-0      0        0.0  
rec-4883-org rec-4883-dup-0      1        1.0  
rec-66-org   rec-3318-dup-0      1        0.0  
             rec-66-dup-0        1        1.0  
             rec-3838-dup-0      0        0.0  

[77249 rows x 6 columns]

In [ ]:
features.describe()

,given_name,surname,date_of_birth,suburb,state,address_1
count,77249.0,77249.000000,77249.000000,77249.000000,77249.000000,77249.000000
mean,1.0,0.044428,0.037929,0.032259,0.248767,0.036700
std,0.0,0.206045,0.191027,0.176689,0.432301,0.188024
min,1.0,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.0,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.0,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.0,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.0,1.000000,1.000000,1.000000,1.000000,1.000000


The last step is to decide which records belong to the same person. In
this example, we keep it simple:

In [ ]:
features.sum(axis=1).value_counts().sort_index(ascending=False)

6.0     1566
5.0     1332
4.0      343
3.0      146
2.0    16427
1.0    57435
Name: count, dtype: int64

In [ ]:
features[features.sum(axis=1) > 3]

,,given_name,surname,date_of_birth,suburb,state,address_1
rec_id_1,rec_id_2,,,,,,
rec-1016-org,rec-1016-dup-0,1,1.0,1,1,0,1.0
rec-4405-org,rec-4405-dup-0,1,1.0,1,1,1,1.0
rec-1288-org,rec-1288-dup-0,1,1.0,1,0,1,1.0
rec-3585-org,rec-3585-dup-0,1,1.0,1,1,1,1.0
rec-298-org,rec-298-dup-0,1,1.0,1,1,1,0.0
...,...,...,...,...,...,...,...
rec-2613-org,rec-2613-dup-0,1,1.0,1,1,1,1.0
rec-2235-org,rec-2235-dup-0,1,1.0,1,1,1,1.0
rec-3877-org,rec-3877-dup-0,1,1.0,1,1,1,1.0


Full code
---------

In [ ]:
import recordlinkage
from recordlinkage.datasets import load_febrl4

dfA, dfB = load_febrl4()

# Indexation step
indexer = recordlinkage.Index()
indexer.block("given_name")
candidate_links = indexer.index(dfA, dfB)

# Comparison step
compare_cl = recordlinkage.Compare()

compare_cl.exact("given_name", "given_name", label="given_name")
compare_cl.string("surname", "surname", method="jarowinkler", threshold=0.85, label="surname")
compare_cl.exact("date_of_birth", "date_of_birth", label="date_of_birth")
compare_cl.exact("suburb", "suburb", label="suburb")
compare_cl.exact("state", "state", label="state")
compare_cl.string("address_1", "address_1", threshold=0.85, label="address_1")

features = compare_cl.compute(candidate_links, dfA, dfB)

# Classification step
matches = features[features.sum(axis=1) > 3]
print(len(matches))

3241


Unsupervised learning
---------
In situations without training data, unsupervised learning can be a solution for record linkage problems. In this section, we discuss two unsupervised learning methods. One algorithm is K-means clustering, and the other algorithm is an implementation of the Expectation-Maximisation algorithm. Most of the time, unsupervised learning algorithms take more computational time because of the iterative structure in these algorithms.

Data Preparation
---------
We will use the same datasets (dfA and dfB) to compare the simple sum-up by cell selection of matching pairs with the results of two unsupervised learning algorithms.

Step 1 (Indexing) and Step 2 (Comparison) are performed again.

In [ ]:
from recordlinkage.index import Block
indexer = recordlinkage.Index()
indexer.block('given_name', 'given_name')
candidate_links = indexer.index(dfA, dfB)
len(candidate_links)

77249

In [ ]:
compare_cl = recordlinkage.Compare()
compare_cl.exact("given_name", "given_name", label="given_name")
compare_cl.string("surname", "surname", method="jarowinkler", threshold=0.85, label="surname")
compare_cl.exact("date_of_birth", "date_of_birth", label="date_of_birth")
compare_cl.exact("suburb", "suburb", label="suburb")
compare_cl.exact("state", "state", label="state")
compare_cl.string("address_1", "address_1", threshold=0.85, label="address_1")
features = compare_cl.compute(candidate_links, dfA, dfB)

In [ ]:
features

given_name  surname  date_of_birth  suburb  \
rec_id_1     rec_id_2                                                     
rec-1070-org rec-3024-dup-0           1      0.0              0       0   
             rec-2371-dup-0           1      0.0              0       0   
             rec-4652-dup-0           1      0.0              0       0   
             rec-4795-dup-0           1      0.0              0       0   
             rec-1314-dup-0           1      0.0              0       0   
...                                 ...      ...            ...     ...   
rec-1003-org rec-3321-dup-0           1      0.0              0       0   
rec-4883-org rec-4883-dup-0           1      1.0              1       1   
rec-66-org   rec-3318-dup-0           1      0.0              0       0   
             rec-66-dup-0             1      1.0              1       1   
             rec-3838-dup-0           1      0.0              0       0   

                             state  address_1  
rec_id_1     rec_id_2                          
rec-1070-org rec-3024-dup-0      1        0.0  
             rec-2371-dup-0      0        0.0  
             rec-4652-dup-0      0        0.0  
             rec-4795-dup-0      1        0.0  
             rec-1314-dup-0      1        0.0  
...                            ...        ...  
rec-1003-org rec-3321-dup-0      0        0.0  
rec-4883-org rec-4883-dup-0      1        1.0  
rec-66-org   rec-3318-dup-0      1        0.0  
             rec-66-dup-0        1        1.0  
             rec-3838-dup-0      0        0.0  

[77249 rows x 6 columns]

K-means clustering
---------
The K-means clustering algorithm is well-known and widely used in big data analysis. The K-means classifier in the Python Record Linkage Toolkit package is configured in such a way that it can be used for linking records.

In [ ]:
kmeans = recordlinkage.KMeansClassifier()
result_kmeans = kmeans.fit_predict(features)
len(result_kmeans)

3241

In [ ]:
result_kmeans

MultiIndex([('rec-1016-org', 'rec-1016-dup-0'),
            ('rec-4405-org', 'rec-4405-dup-0'),
            ('rec-1288-org', 'rec-1288-dup-0'),
            ('rec-3585-org', 'rec-3585-dup-0'),
            ( 'rec-298-org',  'rec-298-dup-0'),
            ('rec-2404-org', 'rec-2404-dup-0'),
            ( 'rec-453-org',  'rec-453-dup-0'),
            ('rec-4866-org', 'rec-4866-dup-0'),
            ('rec-4302-org', 'rec-4302-dup-0'),
            ( 'rec-420-org',  'rec-420-dup-0'),
            ...
            ('rec-3125-org', 'rec-3125-dup-0'),
            ( 'rec-715-org',  'rec-715-dup-0'),
            ( 'rec-674-org',  'rec-674-dup-0'),
            ('rec-1916-org', 'rec-1916-dup-0'),
            ('rec-2792-org', 'rec-2792-dup-0'),
            ('rec-2613-org', 'rec-2613-dup-0'),
            ('rec-2235-org', 'rec-2235-dup-0'),
            ('rec-3877-org', 'rec-3877-dup-0'),
            ('rec-4883-org', 'rec-4883-dup-0'),
            (  'rec-66-org',   'rec-66-dup-0')],
           names=['rec_

Expectation/Conditional Maximization Algorithm
---------
The ECM-algorithm is an Expectation-Maximisation algorithm with some additional constraints. This algorithm is closely related to the Naive Bayes algorithm. The ECM algorithm is also closely related to estimating the parameters in the Fellegi and Sunter (1969) framework. The algorithms assume that the attributes are independent of each other. The Naive Bayes algorithm uses the same principles.

In [ ]:
ecm = recordlinkage.ECMClassifier(binarize=0.8)
result_ecm = ecm.fit_predict(features)
len(result_ecm)

3261

In [ ]:
result_ecm

MultiIndex([('rec-1016-org', 'rec-1016-dup-0'),
            ('rec-4405-org', 'rec-4405-dup-0'),
            ('rec-1288-org', 'rec-1288-dup-0'),
            ('rec-3585-org', 'rec-3585-dup-0'),
            ( 'rec-298-org',  'rec-298-dup-0'),
            ('rec-2404-org', 'rec-2404-dup-0'),
            ( 'rec-453-org',  'rec-453-dup-0'),
            ('rec-4866-org', 'rec-4866-dup-0'),
            ('rec-4302-org', 'rec-4302-dup-0'),
            ( 'rec-383-org',  'rec-383-dup-0'),
            ...
            ('rec-3125-org', 'rec-3125-dup-0'),
            ( 'rec-715-org',  'rec-715-dup-0'),
            ( 'rec-674-org',  'rec-674-dup-0'),
            ('rec-1916-org', 'rec-1916-dup-0'),
            ('rec-2792-org', 'rec-2792-dup-0'),
            ('rec-2613-org', 'rec-2613-dup-0'),
            ('rec-2235-org', 'rec-2235-dup-0'),
            ('rec-3877-org', 'rec-3877-dup-0'),
            ('rec-4883-org', 'rec-4883-dup-0'),
            (  'rec-66-org',   'rec-66-dup-0')],
           names=['rec_